# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con experta

Debes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.

---


* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

### Mision A

In [ ]:
from experta import *

class EstadoObra(Fact):
    """Datos capturados por sensores en el sitio"""
    pass

class MotorSeguridad(KnowledgeEngine):
    # TODO: Implementar las reglas de seguridad aquí
    pass

# Prueba el sistema con viento de 65 km/h y presencia de grietas

### Mision B

In [ ]:
from constraint import *

def planificar_maquinaria():
    problem = Problem()
    # TODO: Definir máquinas (variables) y zonas (dominios)
    # TODO: Aplicar restricciones lógicas
    soluciones = problem.getSolutions()
    print(f"Configuraciones seguras encontradas: {soluciones}")

planificar_maquinaria()

## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*

# Resolución

## Mision A

In [8]:
!pip install experta
!pip install --upgrade frozendict

  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
experta 1.9.4 requires frozendict==1.2, but you have frozendict 2.4.7 which is incompatible.


In [9]:
from experta import *

class EstadoObra(Fact):
    """Representa los datos sensoriales del sitio de construcción."""
    pass

class MotorSeguridad(KnowledgeEngine):

    # REGLA 1: RIESGO CRÍTICO (Prioridad Máxima)
    @Rule(OR(EstadoObra(viento=lambda v: v > 60),
             EstadoObra(grietas=True)),
          salience=100)
    def riesgo_critico(self):
        print("RESULTADO: RIESGO CRÍTICO.")
        print("ACCIÓN: PARO TOTAL DE OBRA. Evacuar zonas de altura y excavación.")
        self.declare(EstadoObra(estado_final='parado'))

    # REGLA 2: RIESGO MODERADO
    @Rule(AND(NOT(EstadoObra(estado_final='parado')),
              OR(EstadoObra(viento=lambda v: 40 <= v <= 60),
                 EstadoObra(humedad_extrema=True))))
    def riesgo_moderado(self):
        print("RESULTADO: RIESGO MODERADO.")
        print("ACCIÓN: PRECAUCIÓN. Suspender maniobras de izaje pesado.")

    # REGLA 3: BAJO RIESGO
    @Rule(AND(EstadoObra(viento=lambda v: v < 40),
              EstadoObra(grietas=False),
              EstadoObra(humedad_extrema=False)))
    def bajo_riesgo(self):
        print("RESULTADO: BAJO RIESGO.")
        print("ACCIÓN: OPERACIÓN NORMAL. Continuar según cronograma.")

# --- PRUEBA DEL SISTEMA ---
engine = MotorSeguridad()
engine.reset()

# Caso de prueba: Viento fuerte y detección de grietas
engine.declare(EstadoObra(viento=65, grietas=True, humedad_extrema=False))
engine.run()

RESULTADO: RIESGO CRÍTICO.
ACCIÓN: PARO TOTAL DE OBRA. Evacuar zonas de altura y excavación.


## Mision B

In [10]:
!pip install python-constraint

  Preparing metadata (setup.py) ... done
  Created wheel for python-constraint: filename=python_constraint-1.4.0-py2.py3-none-any.whl size=24061 sha256=94c66dc092b1af6a8df784ff96f660ef508e6a769339d5c74f6ce4f6854334df
  Stored in directory: /root/.cache/pip/wheels/c1/d2/3d/082849b61a9c6de02d4a7c8a402c224640f08d8a971307b92b
Successfully built python-constraint


In [11]:
from constraint import *

def planificar_maquinaria():
    problem = Problem()

    # 1. Definimos Variables (Máquinas) y Dominios (Zonas)
    maquinas = ["Grua_Torre", "Excavadora", "Hormigonera"]
    zonas = ["Zona_Estable", "Zona_Estrecha", "Zona_Comun"]

    problem.addVariables(maquinas, zonas)

    # 2. REGLAS DE ORO (Restricciones)

    # Restricción 1: Exclusividad (Una máquina por zona)
    problem.addConstraint(AllDifferentConstraint())

    # Restricción 2: Grúa Torre solo en Zona Estable (Base reforzada)
    problem.addConstraint(lambda gt: gt == "Zona_Estable", ["Grua_Torre"])

    # Restricción 3: Excavadora NO entra en Zona Estrecha
    problem.addConstraint(lambda ex: ex != "Zona_Estrecha", ["Excavadora"])

    # Restricción 4: Hormigonera NO puede estar con la Grúa (Congestión)
    # Al usar AllDifferent, ya están en zonas distintas, pero aquí
    # reforzamos que no pueden compartir recursos logísticos.
    problem.addConstraint(lambda ho, gt: ho != gt, ("Hormigonera", "Grua_Torre"))

    # 3. Resolución del Espacio del Problema
    soluciones = problem.getSolutions()

    print(f"Se encontraron {len(soluciones)} configuraciones seguras:")
    for i, sol in enumerate(soluciones, 1):
        print(f"Opción {i}: {sol}")

planificar_maquinaria()

Se encontraron 1 configuraciones seguras:
Opción 1: {'Grua_Torre': 'Zona_Estable', 'Hormigonera': 'Zona_Estrecha', 'Excavadora': 'Zona_Comun'}
